# Экспертная валидация извлечения изменений из НПА (Google Colab)

Блокнот заменяет агентский конвейер [`legal-agent-harness`](https://github.com/lawful-good-project/legal-agent-harness)
на офлайн-процедуру для эксперта: **загрузить документ → Run all → оценить результат в форме → отправить в Hugging Face**.

1. Эксперт нажимает в блокноте **«Загрузить документ»** (файл или ссылка) — блокнот сам ждёт загрузку и продолжает работу. HF-токен вводится в защищённое поле формы.
2. Блокнот извлекает текст документа и через **бесплатную LLM (Pollinations, без ключа)** строит черновик `records` + граф «изменяющий акт → изменённая норма» по профилю amendment.
3. Генерируется **автономная офлайн HTML-форма** (`expert_review.html`): результаты извлечения, поля для оценки эксперта и отправка записи через HF API.
4. Запись публикуется в датасет `lawful-good-project/legal-agent-calibrations`.

> Это учебная проверка качества извлечения, а не юридическая консультация. Никакая оценка не выставляется автоматически — только экспертом.

**Перед запуском** в Colab откройте слева **🔑 Секреты (Secrets)** и добавьте `HF_TOKEN` с правом записи в датасет. Включите «Notebook access» для этого блокнота. Токен не нужно вставлять в код и не будет напечатан в выводе.

**Если форма не нужна в браузере:** любую ссылку на официальный документ (PDF/DOCX/TXT) можно указать в параметре `URLS` — блокнот скачает её сам.

## 1. Установка зависимостей

Ставим `pypdf` — для PDF с текстовым слоем. LLM вызывается через стандартный `urllib` из stdlib, сторонних SDK не требуется.

In [ ]:
%pip install -q -U pypdf g4f
print("Dependencies ready: pypdf, g4f")


Сначала короткими запросами проверяются доступные модели g4f; выбирается первая ответившая. Если ни одна не отвечает, блокнот перебирает резервные бесплатные источники Pollinations (`gpt-oss-20b`, затем `openai`). HTML создаётся только после валидного JSON-ответа.


In [ ]:
import asyncio
import os, json, datetime as dt

MODE = "single"          # "single" или "batch"
URLS = ""                # для batch: ссылки через запятую; для single — можно оставить пустым
HF_REPO_ID = "lawful-good-project/legal-agent-calibrations"
HF_TOKEN_SECRET_NAME = "HF_TOKEN"   # имя секрета Colab; явно в коде токен не хранится

MODEL = "gpt-4o"                  # g4f public-provider route
LLM_BASE_URL = "g4f://client"
G4F_MODELS_TO_PROBE = []  # заполнится автоматически из каталога g4f
G4F_CONCURRENCY = 4              # ограничение параллельных запросов
MAX_ATTEMPTS = 4                  # повторов при 402/429/503 (лимит бесплатного тарифа)
MIN_INTERVAL_SEC = 31             # пауза между запросами: бесплатный тариф — 1 запрос / 30 с
TIMEOUT_SEC = 180
MAX_TEXT_CHARS = 24000            # сколько символов документа видит модель (начало + конец)
MAX_UPLOAD_MB = 20

EXPERT_NAME = ""                  # необязательно: подпись эксперта в записи
AUTHORITY_HINT = ""               # необязательно: напр. "Государственная Дума ФС РФ"
JURISDICTION = "RU"               # RU | INT | EU | US | UK | CIS | OTHER

# g4f определяет доступность провайдеров непосредственно при экстракции.
# Сначала проверяем доступные модели g4f коротким запросом; сохраняем первую рабочую.
import time, json, datetime as _dt

def _g4f_candidates():
    try:
        from g4f import Provider
        from g4f.models import ModelUtils
        models = getattr(ModelUtils, "convert", {}) or {}
        names = list(models.keys()) if isinstance(models, dict) else []
        if not names:
            names = ["gpt-4o", "gpt-4", "claude-3-haiku", "llama-3.1-70b"]
        return list(dict.fromkeys(str(x) for x in names if x))
    except Exception:
        return ["gpt-4o", "gpt-4", "claude-3-haiku", "llama-3.1-70b"]

def _g4f_ping(model):
    try:
        from g4f.client import Client
        r = Client().chat.completions.create(model=model, messages=[{"role":"user","content":"Reply exactly: OK"}], temperature=0)
        return bool(r and r.choices and (r.choices[0].message.content or "").strip())
    except Exception as exc:
        print(f"  g4f {model}: недоступна ({type(exc).__name__})")
        return False

G4F_MODELS_TO_PROBE = _g4f_candidates()
ACTIVE_G4F_MODEL = None

# Serial probing in the notebook cell: Colab IPython already runs an event loop,
# so asyncio.run() would fail with "cannot be called from a running event loop".
print(f"Проверяю модели g4f: {len(G4F_MODELS_TO_PROBE)}")
for _candidate in G4F_MODELS_TO_PROBE:
    print("  ping", _candidate, "…", end=" ")
    if _g4f_ping(_candidate):
        ACTIVE_G4F_MODEL = _candidate
        print("OK — выбрана")
        break
    print("нет ответа")
if ACTIVE_G4F_MODEL is None:
    print("⚠️ Ни одна модель g4f не прошла ping; будет перебор резервных источников.")
else:
    MODEL = ACTIVE_G4F_MODEL

CFG = dict(mode=MODE, urls=URLS, repo=HF_REPO_ID, model=MODEL, base_url=LLM_BASE_URL,
           max_attempts=MAX_ATTEMPTS, min_interval=MIN_INTERVAL_SEC, timeout=TIMEOUT_SEC,
           max_chars=MAX_TEXT_CHARS, max_mb=MAX_UPLOAD_MB,
           expert=EXPERT_NAME, authority=AUTHORITY_HINT, jurisdiction=JURISDICTION,
           created_at=dt.date.today().isoformat(), run_id="run-" + dt.datetime.now().strftime("%Y%m%d%H%M%S"))
print("активная модель:", MODEL)
print(json.dumps(CFG, ensure_ascii=False, indent=2))

## 3. Загрузка документа

В ячейке ниже появится кнопка **«Загрузить документ»**: выберите файл на устройстве (PDF / DOCX / TXT / HTML / RTF)
или вставьте прямую ссылку. Блокнот **ждёт загрузку**, сохраняет файл в `/content/uploads` и продолжает выполнение
следующих ячеек автоматически — повторять Run all не нужно.

Во время передачи под кнопкой виден **прогресс-бар со статистикой**: процент, «файлов N из M», переданный объём
из общего, скорость и затраченное время. По завершении та же статистика печатается в выводе ячейки.

**Как файл попадает в ядро.** Блокнот поднимает внутри runtime небольшой HTTP-приёмник на `127.0.0.1` (случайный порт,
одноразовый токен в теле запроса). Файл уходит **телом POST напрямую из страницы**, без `invokeFunction`
(на нём крупные файлы замирали: 1.5 МБ превращались в ~4 МБ строки в строке).

Адрес приёмника блокнот не угадывает: вывод Colab живёт в iframe, поэтому виджет собирает абсолютный адрес из
`parent.location` и, если тот недоступен, перебирает `/proxy/<порт>/`, `colab.research.google.com`, `localhost:8080`
и `127.0.0.1`, запоминая первый рабочий. Если не сработал ни один, в форме виден **каждый** адрес с причиной.

Отправка сделана «простым» CORS-запросом: тело — `ArrayBuffer`, токен и имя файла — в query. Это принципиально:
любой `xhr.upload.onprogress` или нестандартный `Content-Type` включает preflight, а его песочница Colab не пропускает.
Поэтому шкала прогресса считается по чтению файла, а не по отправке.

Загружать каждый раз необязательно: если файл уже лежит в Colab или на Google Drive, укажите путь в `LOCAL_PATH`
(Drive сначала подключите: значок 📁 → Mount Drive) — тогда кнопка не появится.

In [ ]:
LOCAL_PATH = ""      # напр. "/content/drive/MyDrive/акт.pdf"; пусто — спросить файл кнопкой
SKIP_UPLOAD = False  # True — не спрашивать, взять готовые файлы из /content
UPLOAD_DIR = "/content/uploads" if os.path.isdir("/content") else "uploads"
UPLOAD_WAIT_SEC = 15 * 60   # сколько ждать файл, прежде чем честно сообщить о проблеме
STALL_CHECK_SEC = 60        # как часто напоминать, что ячейка всё ещё ждёт загрузку
UPLOAD_TIMEOUT_SEC = 45     # сколько браузер ждёт ответа приёмника, прежде чем перейти к след. адресу
_UPLOAD_SERVER = {"srv": None, "token": None, "url": "/upload", "local_url": None, "port": None}
_last_upload_stats = {"files": 0, "bytes": 0, "seconds": 0.0}
ALLOWED_EXT = (".pdf", ".docx", ".txt", ".md", ".html", ".htm", ".rtf")

In [ ]:
import asyncio, base64, glob, http.server, io, json, os, re, threading, time, urllib.parse, urllib.request


def _safe_name(name):
    """Имя файла без путей и опасных символов; формат проверяется по белому списку."""
    base = os.path.basename(str(name or "").replace("\\", "/")).strip()
    base = re.sub(r"[^\w.\- ()\u0400-\u04FF]+", "_", base)[:120]
    ext = os.path.splitext(base)[1].lower()
    if not base or ext not in ALLOWED_EXT:
        raise ValueError(f"формат «{ext or base}» не поддерживается; допустимо: {', '.join(ALLOWED_EXT)}")
    return base


def _store(name, raw):
    if not raw:
        raise ValueError("пустой файл")
    if len(raw) > CFG["max_mb"] * 1024 * 1024:
        raise ValueError(f"файл больше {CFG['max_mb']} МБ")
    # Create the destination on every call: the runtime may have restarted,
    # or the upload helper may be invoked before the setup cell ran.
    os.makedirs(UPLOAD_DIR, exist_ok=True)
    path = os.path.join(UPLOAD_DIR, _safe_name(name))
    with open(path, "wb") as fh:
        fh.write(raw)
    return path


def _download(url):
    req = urllib.request.Request(url, headers={"User-Agent": "legal-review-colab/1.0"})
    with urllib.request.urlopen(req, None, 120) as resp:
        return resp.read(CFG["max_mb"] * 1024 * 1024 + 1)


def _fetch_url(url, name_hint=None):
    """Скачивает документ по ссылке; формат без расширения определит buffer_magic() при сохранении."""
    url = url.strip()
    if not url.lower().startswith(("http://", "https://")):
        raise ValueError("ссылка должна начинаться с http:// или https://")
    if name_hint:
        return _store(name_hint, _download(url))
    name = os.path.basename(urllib.parse.unquote(urllib.parse.urlparse(url).path)) or "document.pdf"
    if os.path.splitext(name)[1].lower() not in ALLOWED_EXT:
        name = os.path.splitext(name)[0] or "document"
    return _store(name, _download(url))


def _js(template, **values):
    """Подставляет __ИМЯ__ в шаблоны. str.replace: метка __TOKEN__ есть и внутри __UPLOAD_TIMEOUT_MS__,
    и в названии самой метки — regex или цепочка replace её бы испортили."""
    for key, value in values.items():
        template = template.replace("__%s__" % key, value)
    return template


def _parse_multipart(body, boundary):
    """Минимальный разбор multipart/form-data: token, name, file (без сторонних библиотек).
    Виджет шлёт тело как есть (байты файла), multipart сюда попадает только от сторонних клиентов."""
    fields, filename = {}, "document.pdf"
    sep = b"--" + boundary
    for part in body.split(sep):
        if not part.strip() or part.strip() == b"--":
            continue
        # разделитель берём как bytes: сравнение с b"\r\n\r\n" вместо строки (раньше str с bytes падал)
        head, delim, data = part.partition(b"\r\n\r\n")
        if not delim:
            continue
        text = head.decode("utf-8", "replace")
        data = data[:-4] if data.endswith(b"\r\n--") else (data[:-2] if data.endswith(b"\r\n") else data)
        name_match = re.search(r'name="([^"]*)"', text)
        field = name_match.group(1) if name_match else ""
        file_match = re.search(r'filename="([^"]*)"', text)
        if file_match:
            filename = urllib.parse.unquote(file_match.group(1)) or filename
            fields["__file__"] = data
        elif field:
            fields[field] = data.decode("utf-8", "replace")
    return fields.get("token"), fields.get("name") or filename, fields.get("__file__", b"")


def _start_upload_server():
    """Поднимает HTTP-приёмник внутри kernel-процесса (проброс портов не нужен)."""
    srv = _UPLOAD_SERVER.get("srv")
    if srv is not None and not getattr(srv, "_upload_closed", False):
        port = srv.server_address[1]
        _UPLOAD_SERVER.update(url=f"/proxy/{port}/receive", local_url=f"http://127.0.0.1:{port}/receive", port=port)
        return srv
    # прошлый запуск закрыл приёмник — старый сокет не оживить, поднимаем новый
    _UPLOAD_SERVER.update(srv=None, token=None, url="/upload", local_url=None, port=None)
    import secrets
    token = secrets.token_urlsafe(16)
    srv = None
    for _ in range(20):
        try:
            srv = http.server.ThreadingHTTPServer(("127.0.0.1", 0), _UploadHandler)
            break
        except OSError:
            time.sleep(0.05)
    if srv is None:
        raise RuntimeError("не удалось открыть локальный порт для загрузки")
    srv.daemon_threads = True
    srv.allow_reuse_address = True
    srv.token = token
    srv.received = 0
    srv.paths = []
    srv.delivered = False
    srv.on_files = lambda paths: None      # подменяется в _upload_form
    srv.last_error = None
    srv._upload_closed = False
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    port = srv.server_address[1]
    # Colab проксирует /proxy/<порт>/ с того же хоста, где открыт блокнот, на 127.0.0.1 внутри VM
    _UPLOAD_SERVER.update(srv=srv, token=token, port=port,
                          url=f"/proxy/{port}/receive",
                          local_url=f"http://127.0.0.1:{port}/receive")
    return srv


In [ ]:
# Use Colab built-in upload transport rather than the unreliable local proxy.
from pathlib import Path
files = []
if LOCAL_PATH:
    files = [LOCAL_PATH]
elif SKIP_UPLOAD:
    files = sorted(sum((glob.glob(f"/content/*{ext}") for ext in ALLOWED_EXT), []))
else:
    try:
        from google.colab import files as colab_files
        uploaded = colab_files.upload()
        for name, data in uploaded.items():
            try:
                dest = _store(name, data)
                files.append(dest)
                print(f"Uploaded {os.path.basename(dest)} ({len(data)/1048576:.2f} MiB)")
            except Exception as exc:
                print(f"Skipped {name}: {type(exc).__name__}: {exc}")
        if not uploaded:
            print("No file selected.")
    except Exception as exc:
        print("Colab upload failed:", type(exc).__name__, str(exc)[:250])
        print("Upload through the left Files panel and set SKIP_UPLOAD = True.")
files = [f for f in dict.fromkeys(files) if os.path.isfile(f)]
if files:
    print(f"\nFiles to process: {len(files)}")
    for f in files:
        print(f"{os.path.basename(f)} ({os.path.getsize(f)/1048576:.2f} MiB)")
else:
    print("No documents uploaded. Rerun this cell or set LOCAL_PATH.")


## 4. Разбор документа в текст

Офлайн-парсинг без OCR: `pypdf` для PDF, `zipfile` для DOCX, регулярные правила для HTML/RTF/TXT.
Хранится только SHA-256 и статистика — сам текст в публикуемую запись не попадает.

In [ ]:
import hashlib, html, io, os, re, zipfile
from pathlib import Path

def _rtf_like(data: bytes) -> bool:
    """Настоящий RTF: заголовок {\\rtf и RTF-экранирование (\\'hh или \\uN), а не готовый UTF-8."""
    head = data.lstrip()[:5].lower().startswith(b"{\\rtf")
    return head and bool(re.search(rb"\\'[0-9a-fA-F]{2}|\\u-?\d+", data[:20000]))


def text_from_bytes(name: str, data: bytes) -> str:
    ext = Path(name).suffix.lower()
    if ext in (".txt", ".md", ".csv", ".xml"):
        for enc in ("utf-8", "cp1251", "koi8-r"):
            try: return data.decode(enc)
            except UnicodeDecodeError: continue
        return data.decode("utf-8", "replace")
    if ext in (".html", ".htm"):
        s = data.decode("utf-8", "replace")
        s = re.sub(r"(?is)<(script|style)[^>]*>.*?</\1>", " ", s)
        return html.unescape(re.sub(r"<[^>]+>", " ", s))
    if ext == ".rtf":
        s = data.decode("latin-1", "replace")
        if _rtf_like(data):
            # RTF: \uN? — unicode, \'hh — cp1251 hex, иначе ASCII
            s = re.sub(r"\\u(-?\d+)\??", lambda m: chr(int(m.group(1)) % 65536), s)
            s = re.sub(r"\\'([0-9a-fA-F]{2})", lambda m: bytes([int(m.group(1), 16)]).decode("cp1251", "replace"), s)
            s = re.sub(r"\\(?:par[d]?|line|tab)\b ?", "\n", s)
            s = re.sub(r"\\[a-z]+-?\d* ?", " ", s)
            return s.replace("{", " ").replace("}", " ")
        for enc in ("utf-8", "cp1251"):
            try: return re.sub(r"^\s*\{\\rtf1[^ ]* ?|\\par?}\s*$", "", data.decode(enc)).replace("\\par", "\n")
            except UnicodeDecodeError: continue
        return data.decode("utf-8", "replace")
    if ext == ".docx":
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            xml = z.read("word/document.xml").decode("utf-8", "replace")
        xml = xml.replace("</w:p>", "\n")
        return html.unescape(re.sub(r"<[^>]+>", "", xml))
    if ext == ".pdf":
        try:
            from pypdf import PdfReader
            reader = PdfReader(io.BytesIO(data))
            pages = [(p.extract_text() or "") for p in reader.pages[:200]]
            text = "\n".join(pages)
            if text.strip(): return text
        except Exception as exc:
            print("pypdf не смог прочитать PDF:", type(exc).__name__)
        # наихудший случай: выдёргиваем читаемые строки из сырых байтов
        s = data.decode("latin-1", "ignore")
        return "\n".join(re.findall(r"[\x20-\x7eА-Яа-яЁё№«»„“”–—]{8,}", s))
    raise ValueError(f"Неподдерживаемый формат: {ext}")


def normalize(text: str) -> str:
    text = text.replace("\xa0", " ").replace("\u00ad", "")
    lines = [re.sub(r"[ \t]+", " ", ln).strip() for ln in text.splitlines()]
    out, blank = [], 0
    for ln in lines:
        if not ln:
            blank += 1
            if blank > 1: continue
        else:
            blank = 0
        out.append(ln)
    return "\n".join(out).strip()


def load_documents():
    docs = []
    for path in files:
        data = Path(path).read_bytes()
        try:
            text = normalize(text_from_bytes(path, data))
        except Exception as exc:
            print(f"✗ {path}: {type(exc).__name__}: {exc}")
            continue
        docs.append({"filename": os.path.basename(path), "data": data, "text": text,
                     "sha256": hashlib.sha256(data).hexdigest()})
    for url in [u.strip() for u in CFG["urls"].split(",") if u.strip()]:
        try:
            import urllib.request
            req = urllib.request.Request(url, headers={"User-Agent": "legal-review-colab/1.0"})
            with urllib.request.urlopen(req, timeout=60) as r:
                data = r.read(CFG["max_mb"] * 1024 * 1024 + 1)
            if len(data) > CFG["max_mb"] * 1024 * 1024:
                raise ValueError("файл больше лимита")
            name = os.path.basename(urllib.parse.urlparse(url).path) or "document.pdf"
            docs.append({"filename": name, "data": data, "text": normalize(text_from_bytes(name, data)),
                         "sha256": hashlib.sha256(data).hexdigest()})
        except Exception as exc:
            print(f"✗ {url}: {type(exc).__name__}")
    return docs

import urllib.parse
DOCS = load_documents()
for d in DOCS:
    print(f"✓ {d['filename']}: {len(d['text']):,} символов, sha256={d['sha256'][:12]}…")
    if len(d["text"]) < 200 or re.search(r"/(Type|Filter|Font|obj)\b", d["text"]):
        print("  ⚠️ Текстовый слой почти пустой — вероятно, скан без OCR или нестандартный PDF. "
              "Предоставьте DOCX/TXT или OCR-версию: иначе извлечение будет бессмысленным.")
if not DOCS:
    print("Нет документов для обработки (см. ячейку 3).")

## 5. Извлечение через бесплатные провайдеры g4f

Текст документа передаётся через библиотеку g4f одному из доступных публичных провайдеров. Доступность и лимиты меняются; после нескольких ошибок форма не сохраняется. HTML создаётся после проверки ответа JSON; отдельные записи без подтверждённого полного фрагмента пропускаются, остальные сохраняются для экспертной проверки.


In [ ]:
import asyncio
import json, re, time, urllib.error, urllib.request

PREDICATES = ["изменена", "дополнена", "введена", "изложена_в_новой_редакции",
              "признана_утратившей_силу", "исключена", "уточнена"]
TODAY = dt.date.today().isoformat()
DATE_RE = re.compile(r"^[0-9]{4}(-[0-9]{2}(-[0-9]{2})?)?$")

# Короткий промпт намеренно: на длинном рассуждающая модель gpt-oss-20b обрывает ответ.
SYSTEM_PROMPT = (
    "Ты извлекаешь правовые факты из российских нормативных актов. Отвечай ТОЛЬКО JSON-объектом, без markdown.\n"
    'Схема: {"source":{"identifier":"официальный идентификатор акта","authority":"орган","jurisdiction":"RU|INT|EU|US|UK|CIS|OTHER",'
    '"doc_type":"statute|code|amending_act|by-law|case_law|international_treaty|official_guidance|other","effective_date":"YYYY-MM-DD|null"},'
    '"records":[{"norm_id":"гк-рф/ст.N[/п.M]","predicate":"изменена|дополнена|введена|изложена_в_новой_редакции|признана_утратившей_силу|исключена|уточнена",'
    '"subject":"что изменено","object":"каким актом","effective_from":"YYYY-MM-DD|null","effective_to":null,'
    '"change_description":"кратко своими словами", "change_ru":"то же описание для обратной совместимости","confidence":0.0,"quote":"короткая дословная цитата","change_excerpt":"полный дословный текст фрагмента изменения с вводной формулой и полным изменяемым текстом","coordinate":"статья, пункт"}]}\n'
    "Правила: для профиля amendment извлекай факты вида «изменяющий акт → тип изменения → статья/пункт изменяемого НПА → дата действия»; predicate строго из списка; quote — короткая точная цитата; change_excerpt — дословная подстрока из переданного текста (блокнот расширит её до всего абзаца); дату действия указывай только если она прямо есть в тексте, "
    "иначе null; если документ не вносит изменений — records=[]. change_ru и quote до 200 символов.\n"
    "Документ может быть сокращён: «…[середина документа пропущена]…» значит, что часть текста не показана. "
    "Не придумывай номера статей и нормы, которых нет в показанном тексте; norm_id строй по закону из документа "
    "(например «гк-рф/ст.1393» для ГК РФ), а не по номеру статьи самого изменяющего закона."
)


def _window_text(text, limit):
    """Текст для модели: если не влезает — берём начало и конец документа.

    У актов с поправками изменения собраны в «Статье 1» после списка изменяющих документов,
    а хвост содержит заключительные статьи, поэтому обрезка только начала отбрасывала главное.
    """
    if len(text) <= limit:
        return text
    marker = "\n\n…[середина документа пропущена]…\n\n"
    budget = max(limit - len(marker), 1000)
    head = int(budget * 0.6)
    tail = budget - head
    return text[:head] + marker + text[-tail:]


class _Truncated(Exception):
    pass


def _clean_json(text):
    s = (text or "").strip()
    for a, b in (("\u201c", '"'), ("\u201d", '"'), ("\u00ab", '"'), ("\u00bb", '"'),
                 ("\u2018", "'"), ("\u2019", "'")):
        s = s.replace(a, b)
    i = s.find("{")
    if i < 0:
        raise ValueError("в ответе модели нет JSON-объекта")
    s = s[i:].rstrip()
    if s.endswith("```"):
        s = s[: s.rindex("```")].rstrip()
    return s


def _parse_structural(s):
    """JSON с двумя послаблениями: незакрытая строка и обрыв ввода (_Truncated)."""
    n = len(s)
    pos = [0]

    def ws():
        while pos[0] < n and s[pos[0]] in " \t\r\n":
            pos[0] += 1

    def string(expect_key):
        pos[0] += 1
        buf = []
        while True:
            if pos[0] >= n:
                raise _Truncated
            ch = s[pos[0]]
            if ch == "\\" and pos[0] + 1 < n:
                nxt = s[pos[0] + 1]
                if nxt in '"\\/bfnrt':
                    buf.append({"b": "\b", "f": "\f", "n": "\n", "r": "\r", "t": "\t"}.get(nxt, nxt))
                    pos[0] += 2
                else:
                    buf.append('"')
                    pos[0] += 1
                continue
            if ch == '"':
                j = pos[0] + 1
                while j < n and s[j] in " \t\r\n":
                    j += 1
                nxt = s[j] if j < n else ""
                if nxt in (":", ",", "}", "]") or (expect_key and nxt == ""):
                    pos[0] += 1
                    return "".join(buf)
                if nxt == "":
                    raise _Truncated
                buf.append('"')          # «сырая» кавычка внутри значения
                pos[0] += 1
                continue
            buf.append(ch)
            pos[0] += 1

    def value():
        ws()
        if pos[0] >= n:
            raise _Truncated
        c = s[pos[0]]
        if c == "{":
            return obj()
        if c == "[":
            return arr()
        if c == '"':
            return string(False)
        start = pos[0]
        while pos[0] < n and s[pos[0]] not in ',:]}]\n':
            pos[0] += 1
        word = s[start:pos[0]].strip()
        if word in ("true", "false", "null"):
            return {"true": True, "false": False, "null": None}[word]
        try:
            return int(word)
        except ValueError:
            try:
                return float(word)
            except ValueError:
                return word

    def obj():
        pos[0] += 1
        out = {}
        ws()
        if pos[0] >= n:
            raise _Truncated
        if s[pos[0]] == "}":
            pos[0] += 1
            return out
        while True:
            ws()
            if pos[0] >= n:
                raise _Truncated
            if s[pos[0]] != '"':
                raise ValueError("ожидался ключ")
            key = string(True)
            ws()
            if pos[0] >= n:
                raise _Truncated
            if s[pos[0]] != ":":
                raise ValueError("ожидалось ':'")
            pos[0] += 1
            out[key] = value()
            ws()
            if pos[0] >= n:
                return out
            if s[pos[0]] == ",":
                pos[0] += 1
                ws()
                if pos[0] < n and s[pos[0]] == "}":
                    pos[0] += 1
                    return out
                continue
            if s[pos[0]] == "}":
                pos[0] += 1
                return out
            raise ValueError("ожидалось ',' или '}'")

    def arr():
        pos[0] += 1
        out = []
        ws()
        if pos[0] < n and s[pos[0]] == "]":
            pos[0] += 1
            return out
        while True:
            out.append(value())
            ws()
            if pos[0] >= n:
                return out
            if s[pos[0]] == ",":
                pos[0] += 1
                ws()
                if pos[0] < n and s[pos[0]] == "]":
                    pos[0] += 1
                    return out
                continue
            if s[pos[0]] == "]":
                pos[0] += 1
                return out
            raise ValueError("ожидалось ',' или ']'")

    return value()


def parse_json_recovering(text):
    """(obj, truncated): переживает markdown, «сырые» кавычки и обрыв ответа модели."""
    s = _clean_json(text)
    try:
        return json.loads(s), False
    except json.JSONDecodeError:
        pass
    try:
        return _parse_structural(s), False
    except _Truncated:
        pass
    for cut in range(len(s) - 1, 0, -1):
        if s[cut] not in "]},":
            continue
        try:
            cand = _parse_structural(s[: cut + 1])
        except (_Truncated, ValueError, json.JSONDecodeError):
            continue
        if isinstance(cand, dict) and isinstance(cand.get("records"), list):
            return cand, True
    raise ValueError("не удалось восстановить JSON из ответа модели")


def _run_coroutine(coro):
    """Run a coroutine even if an event loop is already active (Colab/IPython)."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    import concurrent.futures
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coro).result()


def llm_extract(text, source_hint):
    """Chunked extraction with bounded async g4f requests and record-level tolerance."""
    from g4f.client import Client

    def chunks(src, size=10000):
        parts = re.split(r"(?=(?:^|\n)\s*(?:Статья|СТАТЬЯ)\s+\d+)", src)
        out, cur = [], ""
        for part in parts:
            if cur and len(cur) + len(part) > size:
                out.append(cur); cur = part
            else:
                cur += part
        if cur: out.append(cur)
        return out or [src[i:i+size] for i in range(0, len(src), size)]

    pieces = chunks(text)
    operative = [x for x in pieces if re.search(
        r"(?i)(внести в|внести следующие изменения|изложить в следующей редакции|дополнить|признать утративш|статья\s+1\b)", x)]
    ordered = list(dict.fromkeys(operative + pieces))
    system = (SYSTEM_PROMPT +
        "\nИщи факты вида изменяющий акт → тип изменения → статья/пункт нормы → дата действия. "
        "Для каждой записи обязательно верни change_excerpt: полный дословный законченный фрагмент "
        "изменения из переданного фрагмента, без пересказа и сокращения. Если изменений нет, records=[].")
    candidates = []
    if ACTIVE_G4F_MODEL: candidates.append(("g4f", ACTIVE_G4F_MODEL))
    candidates.extend(("g4f", m) for m in G4F_MODELS_TO_PROBE if m != ACTIVE_G4F_MODEL)
    candidates.extend([("pollinations", "gpt-oss-20b"), ("pollinations", "openai")])

    def request_sync(backend, model, user):
        if backend == "g4f":
            r = Client().chat.completions.create(
                model=model,
                messages=[{"role":"system", "content":system}, {"role":"user", "content":user}],
                temperature=0)
            return ((r.choices[0].message.content or "").strip()
                    if r and r.choices else "")
        payload = {"model": model, "temperature": 0,
                   "messages": [{"role":"system", "content":system}, {"role":"user", "content":user}]}
        req = urllib.request.Request(
            "https://text.pollinations.ai/openai", data=json.dumps(payload).encode(), method="POST",
            headers={"Content-Type":"application/json", "User-Agent":"legal-review-colab/1.0"})
        with urllib.request.urlopen(req, timeout=CFG["timeout"]) as resp:
            data = json.loads(resp.read().decode())
        return ((data.get("choices") or [{}])[0].get("message") or {}).get("content", "").strip()

    async def request_async(backend, model, user):
        # Prefer g4f's async client; retain a thread fallback for providers without async support.
        if backend == "g4f":
            try:
                from g4f.client import AsyncClient
                r = await AsyncClient().chat.completions.create(
                    model=model,
                    messages=[{"role":"system", "content":system}, {"role":"user", "content":user}],
                    temperature=0)
                return ((r.choices[0].message.content or "").strip()
                        if r and r.choices else "")
            except (ImportError, AttributeError, TypeError, NotImplementedError):
                pass
        return await asyncio.to_thread(request_sync, backend, model, user)

    async def one_piece(idx, piece, sem):
        user = (f"Фрагмент {idx}/{len(ordered)}. Реквизиты: {json.dumps(source_hint, ensure_ascii=False)}\n"
                f"ТЕКСТ ФРАГМЕНТА:\n{piece}\n\nВерни JSON по схеме. Если релевантных изменений нет, records=[] допустим.")
        failures = []
        async with sem:
            for backend, model in candidates:
                try:
                    content = await request_async(backend, model, user)
                    if not content: raise ValueError("empty content")
                    result, was_truncated = parse_json_recovering(content)
                    if not isinstance(result, dict) or not isinstance(result.get("records"), list):
                        raise ValueError("missing records list")
                    print(f"  ✓ fragment {idx}/{len(ordered)}: response received from {backend}/{model}; records={len(result['records'])}")
                    return idx, result, was_truncated, None
                except Exception as exc:
                    failures.append(f"{backend}/{model}: {type(exc).__name__}: {str(exc)[:100]}")
            return idx, {"source": {}, "records": []}, False, "; ".join(failures[-4:])

    async def run_all():
        sem = asyncio.Semaphore(max(1, int(G4F_CONCURRENCY)))
        return await asyncio.gather(*(one_piece(i, piece, sem) for i, piece in enumerate(ordered, 1)))

    # Colab executes this cell synchronously; use a single bounded event loop.
    results = _run_coroutine(run_all())
    source, all_records, errors, truncated = {}, [], [], False
    for idx, result, was_truncated, error in sorted(results, key=lambda x: x[0]):
        if error:
            errors.append(f"fragment {idx}: {error}")
            print(f"  ⚠ fragment {idx}: skipped after provider retries")
            continue
        if not source and isinstance(result.get("source"), dict): source = result["source"]
        all_records.extend(result.get("records") or [])
        truncated = truncated or was_truncated
    unique, seen = [], set()
    for r in all_records:
        key = (str(r.get("norm_id", "")), str(r.get("predicate", "")), str(r.get("quote", "")), str(r.get("change_excerpt", "")))
        if key not in seen:
            seen.add(key); unique.append(r)
    if not unique and re.search(r"(?i)(внести в федеральный закон|внести следующие изменения|изложить в следующей редакции)", text):
        raise RuntimeError("Не найдено ни одной записи; HTML-форма не сохранена")
    if errors: print(f"  ⚠ пропущено фрагментов: {len(errors)}")
    print(f"  ✓ Экстракция завершена: {len(unique)} уникальных записей из {len(ordered)} фрагментов")
    return {"source": source, "records": unique}, truncated

def _predict(raw):
    r = (raw or "").lower()
    if "дополн" in r: return "дополнена"
    if "вве" in r or "введ" in r: return "введена"
    if "исключ" in r or "отмен" in r: return "исключена"
    if "утратив" in r: return "признана_утратившей_силу"
    if "новой редакц" in r: return "изложена_в_новой_редакции"
    if "уточн" in r: return "уточнена"
    return "изменена"


def _norm_id(subject, article):
    a = (subject or "").upper()
    law = "гк-рф" if "ГК" in a else "нпа"
    return f"{law}/ст.{article}" if article else f"{law}/ст.не определена"


def rule_extract(text, filename, authority, jurisdiction):
    """Детерминированный фолбэк: базовые правила без сети."""
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
    date_m = re.search(r"(?<!\d)(20\d{2})[.\-/](\d{1,2})(?:[.\-/](\d{1,2}))?", text)
    date = None
    if date_m:
        d = date_m.groups()
        date = f"{d[0]}-{int(d[1]):02d}" + (f"-{int(d[2]):02d}" if d[2] else "")
    pred_re = re.compile(r"(изложить|изложен\w*|изменить|изменен\w*|дополнить|дополнен\w*|ввести|введен\w*|"
                         r"исключить|исключен\w*|признать[^.]{0,60}утративш\w*|уточнить|уточнен\w*)", re.I)
    art_re = re.compile(r"(?:стать[яиюе]\s*|ст\.\s*)(\d+(?:\.\d+)?)", re.I)
    records = []
    for i, line in enumerate(text.splitlines()):
        am, pm = art_re.search(line), pred_re.search(line)
        if not (am or pm):
            continue
        article = am.group(1) if am else None
        records.append({
            "norm_id": _norm_id(line, article),
            "predicate": _predict(pm.group(1) if pm else ""),
            "subject": f"статья {article}" if article else "не определено",
            "object": filename,
            "effective_from": date, "effective_to": None,
            "change_ru": line[:400],
            "confidence": 0.75 if (am and pm) else 0.4,
            "quote": line[:400],
            "coordinate": f"строка {i + 1}",
        })
        if len(records) >= 25:
            break
    return {"source": {"title": filename, "identifier": filename, "authority": authority,
                       "jurisdiction": jurisdiction, "doc_type": "amending_act",
                       "effective_date": date, "url": None},
            "records": records,
            "method": "rules-fallback", "text_sha256": digest}


def date_in_text(value, text):
    """ISO-дата встречается в тексте в любом привычном виде: 2021-01-17, 17.01.2021, 17 января 2021 г."""
    if not value or len(value) < 10:
        return True  # неполная дата — проверку не проводим
    y, m, d = value[:4], value[5:7], value[8:10]
    if re.search(rf"(?<!\d){y}[-./]({m.lstrip('0')}|0?{m})(?:[-./]({d.lstrip('0')}|0?{d}))?(?!\d)", text):
        return True
    if re.search(rf"(?<!\d)({d.lstrip('0')}|0?{d})[-./]({m.lstrip('0')}|0?{m})[-./]{y}(?!\d)", text):
        return True
    months = ["янв", "фев", "мар", "апр", "ма", "июн", "июл", "авг", "сен", "окт", "ноя", "дек"]
    month = months[int(m) - 1]
    return bool(re.search(rf"(?<!\d)({d.lstrip('0')}|0?{d})\s+{month}\w*\s+{y}", text, re.I))


def _date_guard(n, ef, et, source, issues, text):
    """Ловит типовые галлюцинации модели с датами и обнуляет сомнительные значения."""
    if ef == TODAY and not date_in_text(ef, text):
        issues.append(f"запись {n}: дата «{ef}» совпадает с сегодняшней — похоже на ошибку модели, обнулено")
        ef = None
    if ef and source.get("effective_date") and ef < source["effective_date"] and ef != source["effective_date"]:
        diff = (dt.date.fromisoformat(ef[:10]) - dt.date.fromisoformat(source["effective_date"][:10])).days if len(ef) == 10 and len(source["effective_date"]) == 10 else None
        if diff is not None and diff > -7:
            issues.append(f"запись {n}: дата действия {ef} на {abs(diff)} дн. раньше вступления акта в силу — обнулено, проверьте формулу вступления в силу")
            ef = None
        else:
            issues.append(f"запись {n}: дата действия {ef} раньше вступления акта в силу — проверьте даты")
    return ef, et


def _squash(text):
    return re.sub(r"\s+", " ", text)


def _paragraph_at(text, anchor):
    """Find `anchor` in the document and return the whole paragraph around it."""
    if not anchor:
        return ""
    pos = text.find(anchor)
    if pos >= 0:
        start, end = pos, pos + len(anchor)
    else:
        # The model may retype whitespace differently; locate the squashed form and
        # map the hit back to character offsets of the original text.
        target = _squash(anchor).strip()
        squashed = _squash(text)
        hit = squashed.find(target)
        if hit < 0:
            return ""
        positions = [i for i, ch in enumerate(text) if not ch.isspace()]
        first, last = hit, hit + len(target) - 1
        if first >= len(positions) or last >= len(positions):
            return ""
        start, end = positions[first], positions[last] + 1
    head = text.rfind("\n\n", 0, start)
    start = 0 if head < 0 else head + 2
    tail = text.find("\n\n", end)
    end = len(text) if tail < 0 else tail
    para = text[start:end].strip()
    # Without blank-line separation, stop at the next article heading.
    cut = re.search(r"\n\s*(?:Статья|СТАТЬЯ)\s+\d+", para)
    if cut and cut.start() > 0:
        para = para[: cut.start()].strip()
    return para


def validate(raw, doc, source_hint):
    """Приводит ответ модели к публикуемой схеме и помечает проблемы."""
    src = raw.get("source") or {}
    source = {
        "authority": (src.get("authority") or source_hint.get("authority") or "не указан")[:200],
        "jurisdiction": src.get("jurisdiction") if src.get("jurisdiction") in ("RU", "INT", "EU", "US", "UK", "CIS", "OTHER") else source_hint["jurisdiction"],
        "doc_type": src.get("doc_type") if src.get("doc_type") in ("statute", "code", "amending_act", "by-law", "case_law", "international_treaty", "official_guidance", "other") else "other",
        "title": (src.get("title") or doc["filename"])[:300],
        "identifier": (src.get("identifier") or doc["filename"])[:120],
        "effective_date": src.get("effective_date") if DATE_RE.match(str(src.get("effective_date") or "")) else None,
        "coordinate": "не определена",
        "url": src.get("url") or None,
    }
    issues, records, raw_records = [], [], []
    for n, r in enumerate(raw.get("records") or [], 1):
        pred = str(r.get("predicate") or "").strip()
        if pred not in PREDICATES:
            fixed = _predict(pred)
            issues.append(f"запись {n}: predicate «{pred}» приведён к каноническому «{fixed}» — сверьте вручную")
            pred = fixed
        ef, et = r.get("effective_from"), r.get("effective_to")
        ef = ef if DATE_RE.match(str(ef or "")) else None
        et = et if DATE_RE.match(str(et or "")) else None
        if ef and et and et < ef:
            issues.append(f"запись {n}: effective_to раньше effective_from — обнулено")
            et = None
        ef, et = _date_guard(n, ef, et, source, issues, doc["text"])
        try: conf = max(0.0, min(1.0, float(r.get("confidence"))))
        except (TypeError, ValueError): conf = 0.3
        record = {
            "norm_id": re.sub(r"\s+", "", str(r.get("norm_id") or ""))[:160] or "нпа/ст.не определена",
            "predicate": pred,
            "subject": (str(r.get("subject") or "не определено"))[:300],
            "object": (str(r.get("object") or source["identifier"]))[:500],
            "effective_from": ef, "effective_to": et, "confidence": round(conf, 2),
            "change_description": (str(r.get("change_description") or r.get("change_ru") or ""))[:2000] or None,
            "change_ru": (str(r.get("change_ru") or r.get("change_description") or ""))[:2000] or None,
            "note_ru": None,
            "quote": (str(r.get("quote") or ""))[:500],
            "change_excerpt": str(r.get("change_excerpt") or r.get("full_quote") or "").strip(),
            "coordinate": (str(r.get("coordinate") or "не определена"))[:200],
        }
        records.append(record)
        raw_records.append(r)
        # дословная проверка цитаты по нормализованному тексту
        q = re.sub(r"\s+", " ", record["quote"]).strip()
        normalized_doc = re.sub(r"\s+", " ", doc["text"])
        if len(q) >= 25 and normalized_doc.find(q) == -1:
            issues.append(f"запись {n}: короткая цитата не найдена дословно — проверьте вручную")
        para = _paragraph_at(doc["text"], record["change_excerpt"]) or _paragraph_at(doc["text"], record["quote"])
        if len(_squash(para).strip()) < 40:
            # Neither the excerpt nor the short quote occurs in the source text.
            records.pop()
            raw_records.pop()
            issues.append(f"запись {n}: пропущена — фрагмент не найден дословно в тексте")
            continue
        record["change_excerpt"] = para
        record["paragraph"] = para
        snippet = _squash(record["quote"]).strip()
        if len(snippet) >= 25 and snippet not in _squash(para):
            issues.append(f"запись {n}: короткая цитата не содержится в найденном абзаце — проверьте вручную")
    if records:
        source["coordinate"] = records[0]["coordinate"]
    if source["effective_date"] and not date_in_text(source["effective_date"], doc["text"]):
        issues.insert(0, f"дата вступления в силу {source['effective_date']} не найдена дословно в тексте — проверьте по официальному источнику")
    if raw.get("method") == "rules-fallback":
        issues.extend(_coverage_notes(doc["text"], raw_records))
    return {"source": source, "records": records, "issues": issues}


def _coverage_notes(text, found):
    """Контроль полноты для офлайн-правил: сколько абзацев с «ст. N» не попало в результат."""
    covered = " ".join(str(r.get("quote") or "") for r in found)
    total, missing = 0, []
    for line in text.splitlines():
        if re.search(r"(?:стать[яиюе]\s*|ст\.\s*)\d", line, re.I):
            total += 1
            if line[:120] not in covered:
                missing.append(line.strip()[:90])
    notes = []
    if missing:
        notes.append(f"правила пропустили {len(missing)} из {total} строк с упоминанием статьи — проверьте полноту вручную")
        notes.extend("возможно пропущено: " + m for m in missing[:4])
    return notes


def build_record(doc, extracted, source):
    """Граф «изменяющий акт → норма» + публикуемая запись (без полного текста)."""
    act_id = "act:" + doc["sha256"][:12]
    nodes = [{"id": act_id, "type": "amending_act", "label": source["identifier"]}]
    edges = []
    for r in extracted["records"]:
        norm_node = "norm:" + r["norm_id"]
        if norm_node not in {n["id"] for n in nodes}:
            nodes.append({"id": norm_node, "type": "norm", "label": r["subject"]})
        edges.append({"from": act_id, "to": norm_node, "relation": r["predicate"],
                      "evidence": r.get("change_excerpt") or r["quote"], "coordinate": r["coordinate"]})
    return {
        "schema": 1,
        "schema_version": 1,
        "grading": "pending",
        "attempt": 1,
        "profile": "amendment",
        "document": {"filename": doc["filename"], "sha256": doc["sha256"],
                     "size_bytes": len(doc["data"]), "chars": len(doc["text"])},
        "source": source,
        "nodes": nodes, "edges": edges,
        "records": extracted["records"],
        "issues": extracted["issues"],
        "method": extracted.get("method", f"llm:{CFG['model']}"),
    }


EXTRACTED = []
ALLOW_RULES_FALLBACK = False   # HTML-форма создаётся только после валидного ответа g4f
if DOCS:
    source_hint = {"authority": CFG["authority"], "jurisdiction": CFG["jurisdiction"]}
    for doc in DOCS:
        print(f"\n=== {doc['filename']}")
        extracted = None
        for attempt in (1, 2):
            try:
                raw, truncated = llm_extract(doc["text"], source_hint)
                extracted = validate(raw, doc, source_hint)
                extracted["method"] = f"llm:{CFG['model']}"
                if truncated:
                    extracted["issues"].insert(0, "ОТВЕТ МОДЕЛИ ОБОРВАН: сохранены не все записи — проверьте полноту по документу")
                total = len(doc["text"])
                if total > CFG["max_chars"]:
                    share = CFG["max_chars"] / total
                    extracted["issues"].insert(
                        0, f"ДОКУМЕНТ БОЛЬШОЙ: модели показано {CFG['max_chars']:,} из {total:,} символов "
                           f"({share:.0%}) — начало и конец. Часть статей не анализировалась: проверьте полноту "
                           f"вручную или обработайте документ по частям")
                print(f"  модель вернула записей: {len(extracted['records'])}")
                if not extracted["records"] and extracted.get("skipped"):
                    print(f"  ⚠️ все записи отклонены проверкой цитат: {len(extracted['skipped'])} — форма по этому документу не создаётся")
                    extracted = None
                break
            except Exception as exc:
                print(f"  попытка {attempt} не удалась: {str(exc)[:200]}")
        if extracted is None:
            print("  ✗ эндпоинт недоступен: данные НЕ извлечены. Запустите ячейку 5 позже (бесплатный тариф бывает перегружен).")
            if not ALLOW_RULES_FALLBACK:
                print("  Для черновика на регулярных правилах (низкая точность) установите ALLOW_RULES_FALLBACK = True.")
                continue
            extracted = validate(rule_extract(doc["text"], doc["filename"], source_hint["authority"], source_hint["jurisdiction"]), doc, source_hint)
            extracted["method"] = "rules-fallback"
            extracted["issues"].insert(0, "МОДЕЛЬ НЕ ОТВЕТИЛА: данные извлечены регулярными правилами — точность низкая, проверяйте каждое поле")
        if not extracted["records"]:
            print("  ⚠️ Модель ответила валидным records=[], форму сохраняем с явной пустой выборкой")
        for iss in extracted["issues"][:6]:
            print("  •", iss)
        EXTRACTED.append(build_record(doc, extracted, extracted["source"]))
        print("  записей:", len(EXTRACTED[-1]["records"]), "| связей:", len(EXTRACTED[-1]["edges"]))

## 6. Автономная офлайн HTML-форма

Форма — один HTML-файл без внешних ресурсов и сервера. В него встроены данные извлечения
(JSON в `<script type="application/json">`), по каждому полю — «верно / исправить / нет в источнике»,
оценка 0–10, комментарий и кнопка отправки в HF.

Отправка работает напрямую из браузера: `POST https://huggingface.co/api/datasets/{repo}/commit/main?create_pr=1`
с заголовком `Authorization: Bearer <токен>`. HF API поддерживает CORS, а токен живёт только в памяти страницы
(не пишется в localStorage). Если браузер заблокирует запрос, есть кнопка **скачать JSON** — файл затем отправляется
ячейкой 7.

Когда Run all открывает эту форму, Colab может вывести **«Предупреждение о доверенном выводе»**.
Данные извлечения помечены как данные, но Colab всё равно может почистить содержимое ячейки.
В таком случае выполните ячейку 9, откройте `expert_review.html` из панели файлов Colab
или форкните эту ячейку и выберите «Доверять» — форма при этом не изменится.

In [ ]:
import html, json, re

FORM_HTML = r"""<!doctype html>
<html lang="ru">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Экспертная проверка — извлечение изменений НПА</title>
<style>
  :root { --bg:#f7f7f5; --card:#fff; --ink:#1c1c1e; --muted:#6b6b70; --line:#e3e1dc;
          --ok:#1a7f4b; --warn:#b4690e; --bad:#b3202a; --accent:#25335a; }
  * { box-sizing:border-box; }
  body { margin:0; background:var(--bg); color:var(--ink);
         font:15px/1.55 -apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Helvetica,Arial,sans-serif; }
  header { background:var(--accent); color:#fff; padding:22px 28px; }
  header h1 { margin:0 0 4px; font-size:21px; font-weight:600; }
  header p { margin:0; opacity:.85; font-size:13px; }
  main { max-width:1080px; margin:0 auto; padding:22px 18px 80px; }
  .card { background:var(--card); border:1px solid var(--line); border-radius:10px; padding:18px 20px; margin-bottom:18px; }
  .card h2 { margin:0 0 12px; font-size:16px; font-weight:600; }
  .card h2 .tag { font-size:12px; font-weight:500; color:var(--muted); }
  .grid { display:grid; grid-template-columns:220px 1fr; gap:6px 14px; font-size:14px; }
  .grid dt { color:var(--muted); }
  .grid dd { margin:0; word-break:break-word; }
  .muted { color:var(--muted); font-size:13px; }
  .banner { border-radius:10px; padding:12px 16px; margin-bottom:18px; font-size:14px; border:1px solid; }
  .banner.bad { background:#fdf1f1; border-color:#f2c6c8; color:var(--bad); }
  .banner.warn { background:#fdf6ea; border-color:#f0d9b5; color:var(--warn); }
  .banner.ok { background:#eef8f1; border-color:#bfe3cd; color:var(--ok); }
  .badge { display:inline-block; padding:1px 8px; border-radius:20px; font-size:12px; border:1px solid var(--line); }
  .badge.warn { color:var(--warn); border-color:#f0d9b5; background:#fdf6ea; }
  .badge.bad { color:var(--bad); border-color:#f2c6c8; background:#fdf1f1; }
  .badge.ok { color:var(--ok); border-color:#bfe3cd; background:#eef8f1; }
  .rec { border:1px solid var(--line); border-radius:8px; padding:14px; margin-bottom:12px; }
  .rec-head { display:flex; justify-content:space-between; gap:10px; align-items:baseline; margin-bottom:10px; }
  .rec-head b { font-size:15px; }
  .field { border-top:1px dashed var(--line); padding:10px 0; }
  .field .label { font-size:12px; color:var(--muted); text-transform:uppercase; letter-spacing:.03em; }
  .field .value { margin:2px 0 8px; word-break:break-word; }
  .field .value.missing { color:var(--bad); font-style:italic; }
  .choice { display:flex; flex-wrap:wrap; gap:6px; margin:2px 0 8px; }
  .choice label { border:1px solid var(--line); border-radius:6px; padding:3px 10px; font-size:13px; cursor:pointer; background:#fafaf9; }
  .choice input { margin-right:6px; }
  .choice label:has(input:checked) { background:#eef1f7; border-color:#c3cde2; }
  input[type=text], input[type=password], textarea, select { width:100%; padding:7px 9px; border:1px solid var(--line);
        border-radius:6px; font:inherit; background:#fff; }
  textarea { min-height:74px; resize:vertical; }
  .fix { display:none; margin-top:6px; }
  .fix.show { display:block; }
  .quote { border-left:3px solid var(--line); padding:4px 0 4px 12px; color:#3d3d42; background:#fbfbfa; }
  .full-excerpt { white-space:pre-wrap; overflow-wrap:anywhere; max-height:65vh; overflow:auto; padding:12px; background:#fbfbfa; border-left:4px solid var(--accent); font:13px/1.6 ui-monospace,Consolas,monospace; }
  .score-row { display:flex; align-items:center; gap:14px; flex-wrap:wrap; }
  .score-row input[type=range] { flex:1; min-width:220px; }
  .score-row output { font-size:26px; font-weight:600; min-width:44px; text-align:center; }
  .scale { display:grid; grid-template-columns:repeat(11,1fr); gap:2px; text-align:center; font-size:11px; color:var(--muted); }
  .tabs { display:flex; gap:6px; margin-bottom:12px; }
  .tabs button { border:1px solid var(--line); background:#fafaf9; border-radius:6px; padding:6px 12px; font:inherit; cursor:pointer; }
  .tabs button.active { background:#fff; border-color:#c3cde2; font-weight:600; }
  table { width:100%; border-collapse:collapse; font-size:13px; }
  th, td { text-align:left; padding:6px 8px; border-bottom:1px solid var(--line); vertical-align:top; }
  th { color:var(--muted); font-weight:500; }
  .actions { display:flex; gap:10px; flex-wrap:wrap; align-items:center; }
  button.primary { background:var(--accent); color:#fff; border:0; border-radius:7px; padding:10px 18px; font:inherit; cursor:pointer; }
  button.secondary { background:#fff; border:1px solid var(--line); border-radius:7px; padding:9px 16px; font:inherit; cursor:pointer; }
  button:disabled { opacity:.5; cursor:not-allowed; }
  .status { margin-top:10px; font-size:14px; }
  .status.ok { color:var(--ok); } .status.bad { color:var(--bad); } .status.warn { color:var(--warn); }
  .required-missing { outline:2px solid #f2c6c8; border-radius:6px; }
  .consent { display:flex; gap:8px; align-items:flex-start; font-size:14px; margin:10px 0; }
  .consent input { margin-top:3px; }
  code { background:#f1f1ef; padding:1px 5px; border-radius:4px; font-size:12px; }
</style>
</head>
<body>
<header>
  <h1>Экспертная проверка извлечения изменений в НПА</h1>
  <p>Учебная калибровка, а не юридическая консультация. Результат извлечения сгенерирован моделью и требует проверки человеком.</p>
</header>
<main>

  <div class="banner" id="modeBanner" style="display:none"></div>

  <div class="card" id="docCard">
    <h2>Документ</h2>
    <dl class="grid" id="docMeta"></dl>
    <p class="muted" style="margin:12px 0 0" id="methodNote"></p>
  </div>

  <div class="card">
    <h2>Реквизиты источника <span class="tag">— сверьте с официальным текстом</span></h2>
    <dl class="grid" id="sourceMeta"></dl>
  </div>

  <div class="card" id="issuesCard" style="display:none">
    <h2>Автоматические предупреждения</h2>
    <ul class="muted" id="issuesList" style="margin:0;padding-left:20px"></ul>
  </div>

  <div class="card">
    <h2>Извлечённые факты <span class="tag" id="recCount"></span></h2>
    <p class="muted">По каждому полю: «верно» — совпадает с источником; «исправить» — укажите правильное значение; «нет в источнике» — лишний факт.</p>
    <div id="records"></div>
    <div id="noRecords" style="display:none" class="muted">Записей не извлечено. Оцените результат и опишите, что должно было быть найдено.</div>
  </div>

  <div class="card">
    <h2>Граф «изменяющий акт → норма»</h2>
    <table id="edges"><thead><tr><th>Из</th><th>Связь</th><th>В</th><th>Координата</th></tr></thead><tbody></tbody></table>
  </div>

  <div class="card">
    <h2>Оценка эксперта</h2>
    <div class="score-row">
      <input type="range" min="0" max="10" step="1" id="score" aria-label="Оценка от 0 до 10">
      <output id="scoreOut">—</output>
    </div>
    <div class="scale"><span>0</span><span>1</span><span>2</span><span>3</span><span>4</span><span>5</span><span>6</span><span>7</span><span>8</span><span>9</span><span>10</span></div>
    <p class="muted">0 — не справился; 1–3 — в основном непригодно; 4–6 — часть изменений, существенные ошибки;
      7–8 — структура верна, отдельные ошибки; 9 — почти полное соответствие; 10 — полностью верно по проверенным полям и связям.</p>
    <div style="margin-top:14px">
      <div class="label muted" style="font-size:12px;text-transform:uppercase;letter-spacing:.03em">Комментарий эксперта (обязателен при оценке ниже 10)</div>
      <textarea id="comments" maxlength="2000" placeholder="Конкретные ошибки: напр. «Неверно predicate: указано “изменена”, а в акте “дополнена”. Дата верна.»"></textarea>
    </div>
    <div style="margin-top:14px">
      <div class="label muted" style="font-size:12px;text-transform:uppercase;letter-spacing:.03em">Эксперт (имя или псевдоним, необязательно)</div>
      <input type="text" id="expert" maxlength="80" placeholder="как подписать запись">
    </div>
  </div>

  <div class="card">
    <h2>Отправка в Hugging Face</h2>
    <p class="muted">Токен используется только для одного запроса из этого браузера и не сохраняется на диске.
      Никогда не вставляйте токен в чат, Git или комментарии. Нужен токен с правом <code>write</code> в датасет.</p>
    <div class="tabs">
      <button type="button" class="active" data-tab="token">Токен</button>
      <button type="button" data-tab="file">Идентификатор из HF</button>
    </div>
    <div id="tab-token">
      <input type="password" id="hfToken" autocomplete="off" placeholder="hf_...">
    </div>
    <div id="tab-file" style="display:none">
      <input type="text" id="reviewId" placeholder="напр. rv-8mvr3-2026-10-28" pattern="[a-z0-9][a-z0-9._-]{2,63}">
      <p class="muted">Если токен нельзя использовать в браузере: нажмите «Скачать запись», а затем отправьте через ячейку
        «Загрузка записи в Hugging Face» в блокноте. Форма в первом запуске создаёт постоянный путь из имени файла и хэша.</p>
    </div>
    <div class="consent">
      <input type="checkbox" id="consent">
      <label for="consent">Я — эксперт и лично проверил(а) показанные факты и граф по официальному источнику.
        Результат будет опубликован публично (оценка, комментарий, реквизиты, хэш файла; без полного текста документа).</label>
    </div>
    <div class="actions">
      <button class="primary" id="send">Отправить результат оценки</button>
      <button class="secondary" id="download">Скачать запись (JSON)</button>
      <button class="secondary" id="reset">Сбросить пометки</button>
    </div>
    <div class="status" id="status" role="status" aria-live="polite"></div>
  </div>

  <p class="muted">Harness: <a href="https://github.com/lawful-good-project/legal-agent-harness">legal-agent-harness</a> ·
     Датасет: <a id="repoLink" target="_blank" rel="noopener">lawful-good-project/legal-agent-calibrations</a></p>
</main>

<script id="review-data" type="application/json">__PAYLOAD__</script>
<script>
(function () {
  "use strict";
  const DATA = JSON.parse(document.getElementById("review-data").textContent);
  const RECORDS = DATA.records || [];
  const REPO = DATA.repo;
  const FIELDS = [
    ["norm_id", "norm_id"],
    ["predicate", "predicate · тип изменения"],
    ["subject", "subject · что изменено"],
    ["object", "object · каким актом"],
    ["effective_from", "effective_from"],
    ["effective_to", "effective_to"],
    ["change_description", "change_description · описание изменения"],
    ["coordinate", "coordinate"],
    ["confidence", "confidence"],
    ["quote", "короткая цитата-подтверждение"],
  ];
  const JUDGEMENTS = [
    ["correct", "верно"],
    ["fixed", "исправить"],
    ["absent", "нет в источнике"],
  ];
  const state = {};
  RECORDS.forEach((r, i) => { state[i] = {}; FIELDS.forEach(([k]) => { state[i][k] = { verdict: "", fix: "" }; }); });

  const esc = (s) => String(s == null ? "" : s).replace(/[&<>"']/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));
  const $ = (id) => document.getElementById(id);
  const show = (v) => (v === null || v === undefined || v === "") ? '<span class="missing">не указано</span>' : esc(v);

  $("repoLink").textContent = REPO;
  $("repoLink").href = "https://huggingface.co/datasets/" + REPO;
  $("docMeta").innerHTML = [
    ["Файл", esc(DATA.document.filename)],
    ["SHA-256", "<code>" + esc(DATA.document.sha256) + "</code>"],
    ["Размер", (DATA.document.size_bytes / 1024).toFixed(0) + " КБ · " + DATA.document.chars.toLocaleString("ru-RU") + " символов текста"],
    ["Запись", esc(DATA.review_id) + " · создана " + esc(DATA.created_at)],
  ].map(([k, v]) => "<dt>" + k + "</dt><dd>" + v + "</dd>").join("");
  $("methodNote").textContent = "Метод извлечения: " + DATA.method + ". Полный текст документа не публикуется.";
  if (String(DATA.method).indexOf("rules") === 0 && (DATA.issues || []).some(x => String(x).indexOf("МОДЕЛЬ НЕ ОТВЕТИЛА") === 0)) {
    const b = $("modeBanner");
    b.className = "banner bad"; b.style.display = "";
    b.textContent = "⚠ Модель не ответила. Ниже — черновик на регулярных правилах: он часто пропускает пункты и " +
      "ошибается в типе изменения. Оценивать его как результат модели нельзя; лучше перезапустить извлечение позже.";
  }

  const S = DATA.source || {};
  $("sourceMeta").innerHTML = [
    ["Название", show(S.title)], ["Идентификатор", show(S.identifier)],
    ["Орган", show(S.authority)], ["Юрисдикция", show(S.jurisdiction)],
    ["Тип документа", show(S.doc_type)], ["Дата вступления в силу", show(S.effective_date)],
  ].map(([k, v]) => "<dt>" + k + "</dt><dd>" + v + "</dd>").join("");

  if ((DATA.issues || []).length) {
    $("issuesCard").style.display = "";
    $("issuesList").innerHTML = DATA.issues.map(x => "<li>" + esc(x) + "</li>").join("");
  }

  $("recCount").textContent = "— " + RECORDS.length + " шт. · граф: " + (DATA.edges || []).length + " связей";
  if (!RECORDS.length) $("noRecords").style.display = "";

  function fieldHtml(ri, key, label) {
    const value = RECORDS[ri][key];
    const st = state[ri][key];
    const choices = JUDGEMENTS.map(([v, t]) =>
      '<label><input type="radio" name="' + key + "-" + ri + '" value="' + v + '">' + t + "</label>").join("");
    const isQuote = key === "quote";
    return '<div class="field" data-field="' + key + "-" + ri + '">' +
      '<div class="label">' + esc(label) + "</div>" +
      '<div class="value ' + (isQuote ? "quote" : "") + '">' + show(value) + "</div>" +
      '<div class="choice">' + choices + "</div>" +
      '<div class="fix"><input type="text" placeholder="Правильное значение ' + esc(key) + '"></div>' +
      "</div>";
  }

  $("records").innerHTML = RECORDS.map((r, i) =>
    '<div class="rec" data-rec="' + i + '">' +
      '<div class="rec-head"><b>' + (i + 1) + ". " + esc(r.subject || r.norm_id || "запись") + "</b>" +
      '<span class="badge">' + esc(r.predicate) + (r.effective_from ? " · " + esc(r.effective_from) : "") + "</span></div>" +
      FIELDS.map(([k, l]) => fieldHtml(i, k, l)).join("") +
      '<div class=field><div class=label>Полный абзац источника с изменением</div><pre class=full-excerpt>' + show(r.change_excerpt) + '</pre></div>' +
    "</div>").join("");

  document.getElementById("records").addEventListener("change", (e) => {
    const t = e.target;
    if (t.type !== "radio") return;
    const key = t.name.slice(0, t.name.lastIndexOf("-")), ri = Number(t.name.slice(t.name.lastIndexOf("-") + 1));
    state[ri][key].verdict = t.value;
    const box = document.querySelector('[data-field="' + key + "-" + ri + '"]');
    const fix = box.querySelector(".fix");
    const input = fix.querySelector("input");
    fix.classList.toggle("show", t.value === "fixed");
    if (t.value === "fixed") { input.focus(); } else { input.value = ""; state[ri][key].fix = ""; }
  });
  document.getElementById("records").addEventListener("input", (e) => {
    if (e.target.tagName !== "INPUT" || e.target.type !== "text") return;
    const box = e.target.closest(".field");
    const key = box.dataset.field.slice(0, box.dataset.field.lastIndexOf("-"));
    const ri = Number(box.dataset.field.slice(box.dataset.field.lastIndexOf("-") + 1));
    state[ri][key].fix = e.target.value;
  });

  $("edges").querySelector("tbody").innerHTML = (DATA.edges || []).map(e =>
    "<tr><td>" + esc(e.from) + "</td><td>" + esc(e.relation) + "</td><td>" + esc(e.to) + "</td><td>" + esc(e.coordinate) + "</td></tr>").join("")
    || '<tr><td colspan="4" class="muted">Связей нет</td></tr>';

  const score = $("score"), scoreOut = $("scoreOut");
  score.value = "-1";
  const onScore = () => { const v = Number(score.value); scoreOut.textContent = v < 0 ? "—" : v; };
  score.addEventListener("input", onScore); onScore();

  document.querySelectorAll(".tabs button").forEach(b => b.addEventListener("click", () => {
    document.querySelectorAll(".tabs button").forEach(x => x.classList.toggle("active", x === b));
    $("tab-token").style.display = b.dataset.tab === "token" ? "" : "none";
    $("tab-file").style.display = b.dataset.tab === "file" ? "" : "none";
  }));

  function toUniversalRecord(r, i) {
    const normId = String(r.norm_id || "нпа/ст.не_определена").trim();
    const parts = normId.split("/");
    const nick = ($("expert").value.trim() || "expert").replace(/[^A-Za-z0-9_-]+/g, "_").slice(0, 20) || "expert";
    const ts = r.effective_from || null;
    return {
      schema: 1,
      record_id: nick + "-amd-" + i,
      profile: "amendment",
      title_ru: (r.subject || r.change_description || normId).slice(0, 160),
      norm: { source_law: parts[0] || normId, article: parts.slice(1).join("/") || null, clause: null, norm_id: normId },
      amending_doc: r.object || "",
      predicate: r.predicate,
      ts: ts,
      granularity: ts && String(ts).length === 10 ? "day" : "year",
      change_ru: r.change_description || r.change_ru || null,
      figures: [],
      before_ru: null,
      after_ru: null,
      source: { file: DATA.document.filename, coord: r.coordinate, doc_type: "npa", license: "official-document", url: null },
      provenance: { extracted_by: $("expert").value.trim() || "expert", extracted_at: (new Date()).toISOString().slice(0, 10), method: "assisted", derivation: "verbatim_quote", tool: DATA.method || "legal-agent-colab" },
      review: { status: "proposed", note_ru: $("comments").value.trim() || "проверено экспертом в форме блокнота" },
      confidence: r.confidence,
      evidence_quote: r.change_excerpt || r.quote || ""
    };
  }

  function buildReview() {
    const sc = Number(score.value);
    const comments = $("comments").value.trim();
    if (!(sc >= 0 && sc <= 10)) return { error: "Выберите оценку от 0 до 10.", focus: score };
    if (sc < 10 && !comments) return { error: "Добавьте комментарий: при оценке ниже 10 ошибки нужно указать явно.", focus: $("comments") };
    if (!$("consent").checked) return { error: "Подтвердите, что проверка выполнена лично вами.", focus: $("consent").parentNode };
    const unchecked = [];
    const fields = {};
    RECORDS.forEach((r, i) => {
      fields[i] = {};
      FIELDS.forEach(([k]) => {
        const st = state[i][k];
        fields[i][k] = { verdict: st.verdict || "unchecked", corrected: st.verdict === "fixed" ? (st.fix || null) : null };
        if (!st.verdict) unchecked.push((i + 1) + "·" + k);
      });
    });
    if (unchecked.length && !confirm("Не отмечено полей: " + unchecked.length + ". Отправить как есть?")) {
      return { cancelled: true };
    }
    const review = {
      schema: 1,
      kind: "expert_review",
      session_id: DATA.review_id,
      session_id: DATA.review_id,
      review_id: DATA.review_id,
      created_at: (new Date()).toISOString().slice(0, 10),
      expert: $("expert").value.trim() || null,
      score: sc,
      comments: comments || "",
      summary: comments || "Экспертная проверка выполнена; см. оценку, записи и граф.",
      expert_approved: true,
      source: DATA.source,
      document: DATA.document,
      method: DATA.method,
      records: RECORDS,
      edges: DATA.edges,
      nodes: DATA.nodes,
      issues: DATA.issues,
      field_reviews: fields,
      unchecked_fields: unchecked.length,
    };
    return { review: review };
  }

  function markError(el) {
    document.querySelectorAll(".required-missing").forEach(x => x.classList.remove("required-missing"));
    if (el && el.focus) { el.focus(); if (el.classList) el.classList.add("required-missing"); }
  }
  function setStatus(text, kind) { const s = $("status"); s.textContent = text; s.className = "status " + (kind || ""); }

  function download(review) {
    const blob = new Blob([JSON.stringify(review, null, 2)], { type: "application/json" });
    const a = document.createElement("a");
    a.href = URL.createObjectURL(blob);
    a.download = DATA.review_id + ".json";
    document.body.appendChild(a); a.click(); a.remove();
    setTimeout(() => URL.revokeObjectURL(a.href), 3000);
  }

  $("download").addEventListener("click", () => {
    $("consent").checked = true; // скачивание ничего не публикует
    const out = buildReview();
    if (out.review) { download(out.review); setStatus("Запись сохранена в файл: " + DATA.review_id + ".json", "ok"); }
    else if (out.error) { setStatus(out.error, "bad"); markError(out.focus); }
  });

  $("reset").addEventListener("click", () => {
    if (!confirm("Сбросить все пометки и комментарий?")) return;
    RECORDS.forEach((r, i) => FIELDS.forEach(([k]) => { state[i][k] = { verdict: "", fix: "" }; }));
    document.querySelectorAll('#records input[type=radio]').forEach(x => x.checked = false);
    document.querySelectorAll('#records input[type=text]').forEach(x => x.value = "");
    document.querySelectorAll('.fix').forEach(x => x.classList.remove("show"));
    $("comments").value = ""; $("expert").value = ""; $("consent").checked = false;
    score.value = "-1"; onScore(); setStatus("Пометки сброшены.");
  });

  $("send").addEventListener("click", async () => {
    const btn = $("send");
    const out = buildReview();
    if (!out.review) { if (out.error) { setStatus(out.error, "bad"); markError(out.focus); } return; }
    const review = Object.assign({}, out.review);
    const token = $("hfToken").value.trim();
    const reviewId = ($("reviewId").value.trim() || DATA.review_id);
    if (!/^[a-z0-9][a-z0-9._-]{2,63}$/.test(reviewId)) { setStatus("Некорректный идентификатор записи.", "bad"); return; }
    if (!/^hf_[A-Za-z0-9]+$/.test(token)) {
      if (!token) {
        setStatus("Токен не указан. Без токена нажмите «Скачать запись» и отправьте файл через блокнот.", "warn");
      } else {
        setStatus("Форма HF-токена неверна (ожидается hf_...). Запись можно скачать и отправить из блокнота.", "bad");
      }
      return;
    }
    review.review_id = reviewId;
    btn.disabled = true; setStatus("Отправка в Hugging Face…", "warn");
    const path = "records.jsonl";
    try {
      const raw = await fetch("https://huggingface.co/datasets/" + REPO + "/raw/main/records.jsonl", {
        cache: "no-store", headers: { "Authorization": "Bearer " + token }
      });
      if (!raw.ok && raw.status !== 404) {
        throw new Error("Не удалось прочитать records.jsonl: HTTP " + raw.status);
      }
      const existing = raw.ok ? await raw.text() : "";
      const lines = RECORDS.map((r, i) => JSON.stringify(toUniversalRecord(r, i + 1))).join("\n");
      const merged = (existing && !existing.endsWith("\n") ? existing + "\n" : existing) + lines + "\n";
      const bytes = new TextEncoder().encode(merged);
      let binary = "";
      for (const byte of bytes) binary += String.fromCharCode(byte);
      const payload = [
        { key: "header", value: { summary: "Add expert review " + reviewId } },
        { key: "file", value: { path: path, content: btoa(binary), encoding: "base64" } }
      ].map(item => JSON.stringify(item)).join("\n") + "\n";
      const res = await fetch("https://huggingface.co/api/datasets/" + REPO + "/commit/main?create_pr=1", {
        method: "POST",
        headers: { "Authorization": "Bearer " + token, "Content-Type": "application/x-ndjson" },
        body: payload,
      });
      if (!res.ok) {
        const text = await res.text();
        setStatus("Hugging Face ответил " + res.status + ": " + text.slice(0, 300) + ". Проверьте права токена. Запись можно скачать.", "bad");
        return;
      }
      const data = await res.json();
      setStatus("Опубликовано: " + REPO + "/" + path + ((data.pullRequestUrl || data.commitUrl) ? " · " + (data.pullRequestUrl || data.commitUrl) : ""), "ok");
      window.localStorage.setItem("legal-review-last-ok", path);
    } catch (err) {
      setStatus("Не удалось отправить (сеть или CORS): " + (err && err.message ? err.message : err) + ". Скачайте запись и отправьте через блокнот.", "bad");
    } finally {
      $("hfToken").value = "";
      btn.disabled = false;
    }
  });
})();
</script>
</body>
</html>"""


def render_form(record: dict, review_id: str, repo: str, method: str) -> str:
    payload = json.dumps({
        "review_id": review_id, "created_at": CFG["created_at"], "repo": repo,
        "method": method, "document": record["document"], "source": record["source"],
        "records": record["records"], "edges": record["edges"], "nodes": record["nodes"],
        "issues": record["issues"],
    }, ensure_ascii=False).replace("</", "<\\/")
    return FORM_HTML.replace("__PAYLOAD__", payload)


def make_review_id(record):
    stem = re.sub(r"[^a-z0-9]+", "-", Path(record["document"]["filename"]).stem.lower()).strip("-")[:20] or "doc"
    return f"rv-{stem}-{record['document']['sha256'][:8]}"[:64]


FORMS = []
for rec in EXTRACTED:
    rid = make_review_id(rec)
    out = Path(f"expert_review_{rid}.html")
    out.write_text(render_form(rec, rid, CFG["repo"], rec["method"]), encoding="utf-8")
    FORMS.append(out)
    print(f"✍️  {out}  ({out.stat().st_size / 1024:.0f} КБ), оценка ещё не выставлена")

# заполняется только после того, как эксперт нажмёт «Скачать запись» / отправит через ячейку 7
SENT = {}


def to_universal_record(record, review, index, nick):
    """Build one amendment entry for dataset records.jsonl (universal-record.schema.json)."""
    norm_id = (record.get("norm_id") or "").strip() or "нпа/ст.не_определена"
    parts = norm_id.split("/")
    source_law = parts[0] if parts else norm_id
    article = "/".join(parts[1:]) if len(parts) > 1 else None
    excerpt = (record.get("change_excerpt") or record.get("quote") or "").strip()
    heading = (record.get("subject") or record.get("change_description") or "")[:120]
    safe_nick = re.sub(r"[^A-Za-z0-9_-]+", "_", (nick or "expert"))[:20] or "expert"
    return {
        "schema": 1,
        "record_id": f"{safe_nick}-amd-{index}",
        "profile": "amendment",
        "title_ru": heading or f"{source_law} · {article or norm_id}",
        "norm": {
            "source_law": source_law,
            "article": article,
            "clause": None,
            "norm_id": norm_id,
        },
        "amending_doc": record.get("object") or "",
        "predicate": record.get("predicate"),
        "ts": record.get("effective_from"),
        "granularity": "day" if len(str(record.get("effective_from") or "")) == 10 else "year",
        "change_ru": record.get("change_description") or record.get("change_ru"),
        "figures": [],
        "before_ru": None,
        "after_ru": None,
        "source": {
            "file": review.get("document", {}).get("filename"),
            "coord": record.get("coordinate"),
            "doc_type": "npa",
            "license": "official-document",
            "url": None,
        },
        "provenance": {
            "extracted_by": nick or "expert",
            "extracted_at": review.get("created_at"),
            "method": "assisted",
            "derivation": "verbatim_quote",
            "tool": review.get("method") or "legal-agent-colab",
        },
        "review": {
            "status": "proposed",
            "note_ru": (review.get("comments") or "")[:1000] or "проверено экспертом в форме блокнота",
        },
        "confidence": record.get("confidence"),
        "evidence_quote": excerpt,
    }


def to_records_jsonl(review, nick):
    records = review.get("records") or []
    lines = [to_universal_record(r, review, i, nick) for i, r in enumerate(records, 1)]
    return "".join(json.dumps(line, ensure_ascii=False) + "\n" for line in lines)



## 7. Открыть форму

Форма офлайн: после этого блока её можно скачать и открыть в любом браузере (кнопка «Скачать» справа от вывода).
Для отправки прямо со страницы нужен интернет — форма ходит в HF API; оценка и пометки при этом не покидают браузер.

In [ ]:
from IPython.display import HTML, display

for out in FORMS:
    print(out.name)
    display(HTML(out.read_text(encoding="utf-8")))

## 8. Публикация записи из блокнота

Если в форме сработала отправка напрямую — этот блок ничего не делает.
Иначе скачайте JSON формы (например `rv-*.json`), перетащите его в панель файлов Colab 📁 (в корень `/content`)
и выполните ячейку:
токен берётся из секрета Colab, не печатается. Запись валидируется (оценка 0–10, комментарий при <10, подтверждение эксперта,
отсутствие токенов в полях), затем отправляется через HF Commit API в `validation/<review_id>.json`.
Повторный запуск уже опубликованную запись не дублирует.

In [ ]:
import base64, json, urllib.request, urllib.error
from pathlib import Path


def get_hf_token():
    token = None
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get(HF_TOKEN_SECRET_NAME)
    except Exception:
        token = os.environ.get(HF_TOKEN_SECRET_NAME)
    if not token:
        print(f"⚠️ Токен не найден. Добавьте секрет {HF_TOKEN_SECRET_NAME} в 🔑 панели Colab (с доступом к блокноту) "
              "или задайте переменную окружения с тем же именем.")
        return None
    if not re.match(r"^hf_[A-Za-z0-9]+$", token.strip()):
        print("⚠️ Значение секрета не похоже на HF-токен (ожидается hf_...).")
        return None
    return token.strip()


def validate_review(rv):
    def leak_problem(node, path="root"):
        if isinstance(node, dict):
            for k, v in node.items():
                key = str(k).lower()
                if re.search(r"(^|_)(token|secret|api_key|apikey|password|passwd)($|_)", key):
                    return f"в записи найдено поле секрета: {path}.{k}"
                bad = leak_problem(v, f"{path}.{k}")
                if bad: return bad
        elif isinstance(node, list):
            for i, v in enumerate(node):
                bad = leak_problem(v, f"{path}[{i}]")
                if bad: return bad
        elif isinstance(node, str) and re.search(r"\bhf_[A-Za-z0-9]{10,}", node):
            return f"в записи найдено значение, похожее на HF-токен: {path}"
        return None
    if rv.get("kind") != "expert_review": return "не запись экспертной оценки"
    if not isinstance(rv.get("session_id"), str) or not rv.get("session_id"): return "нет session_id"
    if not (isinstance(rv.get("score"), int) and 0 <= rv["score"] <= 10): return "оценка должна быть целым числом 0–10"
    if rv["score"] < 10 and not rv.get("comments"): return "при оценке ниже 10 нужен комментарий"
    if not isinstance(rv.get("comments"), str): return "comments должен быть строкой"
    if not isinstance(rv.get("summary"), str) or not rv.get("summary"): return "summary должен быть непустой строкой"
    if not isinstance(rv.get("comments"), str): return "comments должен быть строкой"
    if not isinstance(rv.get("summary"), str) or not rv.get("summary"): return "summary должен быть непустой строкой"
    if rv.get("expert_approved") is not True: return "нет явного подтверждения эксперта"
    if not re.match(r"^[a-z0-9][a-z0-9._-]{2,63}$", str(rv.get("review_id") or "")): return "некорректный review_id"
    return leak_problem(rv)


def publish(review, token, repo=CFG["repo"]):
    """Append expert records to the dataset's records.jsonl (universal-record schema)."""
    nick = (review.get("expert") or CFG.get("expert") or "expert").strip()
    body_lines = to_records_jsonl(review, nick)
    if not body_lines.strip():
        return False, "нет записей для публикации"
    existing = ""
    try:
        url = f"https://huggingface.co/datasets/{repo}/raw/main/records.jsonl"
        request = urllib.request.Request(url, headers={"Authorization": f"Bearer {token}"})
        with urllib.request.urlopen(request, timeout=60) as resp:
            existing = resp.read().decode("utf-8")
    except urllib.error.HTTPError as exc:
        if exc.code != 404:
            return False, f"Не удалось прочитать records.jsonl: HTTP {exc.code}"
    except Exception as exc:
        return False, f"Не удалось прочитать records.jsonl ({type(exc).__name__})"
    merged = existing if existing.endswith("\n") or not existing else existing + "\n"
    merged += body_lines
    payload = [
        {"key": "header", "value": {"summary": "Add expert review " + str(review.get("review_id") or "review")}},
        {"key": "file", "value": {"path": "records.jsonl", "encoding": "base64",
                                  "content": base64.b64encode(merged.encode("utf-8")).decode("ascii")}},
    ]
    body = "".join(json.dumps(item) + "\n" for item in payload).encode("utf-8")
    req = urllib.request.Request(
        f"https://huggingface.co/api/datasets/{repo}/commit/main?create_pr=1",
        data=body, method="POST",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/x-ndjson"})
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = json.loads(resp.read().decode("utf-8") or "{}")
            return True, f"{resp.status} · {data.get('pullRequestUrl') or data.get('commitUrl') or 'records.jsonl обновлён'}"
    except urllib.error.HTTPError as exc:
        return False, f"HTTP {exc.code}: {exc.read().decode('utf-8', 'replace')[:300]}"
    except Exception as exc:
        return False, f"{type(exc).__name__}: {str(exc)[:200]}"


review_files = sorted(Path("/content").glob("rv-*.json")) + sorted(Path(".").glob("rv-*.json"))
review_files = list(dict.fromkeys(review_files))
if not review_files:
    print("Нет файлов rv-*.json в /content. Сначала скачайте запись из формы.")
else:
    token = get_hf_token()
    for path in review_files:
        if path.name in SENT:
            print(f"✓ {path.name}: уже опубликовано");
            continue
        rv = json.loads(path.read_text(encoding="utf-8"))
        problem = validate_review(rv)
        if problem:
            print(f"✗ {path.name}: {problem}");
            continue
        if not token:
            print(f"… {path.name}: ожидает токен");
            continue
        ok, msg = publish(rv, token)
        print(("✓ " if ok else "✗ ") + f"{path.name}: {msg}")
        if ok:
            SENT[path.name] = msg

## 9. Скачать формы

Помогает забрать HTML-формы и манифест загрузки из Colab. В манифест попадают только реквизиты и хэши, без текста документа и без токена.

In [ ]:
import shutil, zipfile

manifest = [{"filename": rec["document"]["filename"], "sha256": rec["document"]["sha256"],
             "review_id": make_review_id(rec), "method": rec["method"],
             "records": len(rec["records"]), "run_id": CFG["run_id"]} for rec in EXTRACTED]
Path("extraction_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

if FORMS:
    try:
        from google.colab import files  # type: ignore
        for out in FORMS:
            files.download(str(out))
    except Exception:
        pass
    with zipfile.ZipFile("expert_forms.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for out in FORMS:
            z.write(out.name)
    print("zip:", Path("expert_forms.zip").resolve())
print("manifest:", Path("extraction_manifest.json").resolve())
print("Готово. Заполните форму, поставьте оценку 0–10 и отправьте результат.")

## Ограничения и этика

- **Не юридическое заключение.** Блокнот калибрует извлечение, а не проверяет право.
- **g4f** использует публичные бесплатные провайдеры, доступность которых меняется. При сбое блокнот повторяет запрос и сообщает об ошибке; HTML-форма не создаётся без проверенного ответа модели.
- **Большие документы делятся на фрагменты и обрабатываются последовательно.** Если текст длиннее `MAX_TEXT_CHARS`, для каждого фрагмента выполняется отдельный запрос, результаты объединяются и дедуплицируются. Проверено на 53-ФЗ (215 394 символа = 11% документа, 8 из 78 статей): по фрагменту модель делает выводы о документе в целом — такие документы нужно обрабатывать по частям и проверять полноту вручную. В форме это видно как предупреждение о неполноте; эксперт проверяет каждую запись по официальному источнику.
- Модель **рассуждающая и склонна к обрыву ответа**: частично восстановленный ответ помечается предупреждением.
- Качество ниже, чем у платных моделей: в ответах встречаются неверные `norm_id`/`predicate` и пустые цитаты — ровно это эксперт и проверяет в форме.
- **PDF-сканы** без текстового слоя не распознаются (OCR нет). Блокнот предупреждает, если текстовый слой пустой.
- **Публикуется только запись**: реквизиты, извлечённые факты, оценка, комментарий, SHA-256. Полный текст документа и персональные данные в датасет не попадают.
- **Токен** хранится только в секрете Colab или памяти HTML-страницы. Если он попал в чат, Git или логи — отзовите его в Hugging Face и создайте новый.